In [2]:
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder.appName("Practice").getOrCreate()

# Sample data
data = [
    (1, 101, 'India', 'login', 300),
    (2, 102, 'USA', 'purchase', 900),
    (3, 103, 'India', 'purchase', 1200),
    (4, None, 'Canada', 'login', 200),
    (5, 101, 'India', 'logout', 100)
]

# Column names
columns = ['event_id', 'user_id', 'country', 'event_type', 'session_time_sec']

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Show DataFrame
df.show()

+--------+-------+-------+----------+----------------+
|event_id|user_id|country|event_type|session_time_sec|
+--------+-------+-------+----------+----------------+
|       1|    101|  India|     login|             300|
|       2|    102|    USA|  purchase|             900|
|       3|    103|  India|  purchase|            1200|
|       4|   NULL| Canada|     login|             200|
|       5|    101|  India|    logout|             100|
+--------+-------+-------+----------+----------------+



In [3]:
from pyspark.sql import functions as F

purchase_events = df.filter(F.col('event_type') == 'purchase')

purchase_count = purchase_events.count()

print("Total purchase events:", purchase_count)

Total purchase events: 2


In [5]:
# Sample users data
users_data = [
    (101, 'Neel'),
    (102, 'John'),
    (103, 'Priya')
]

# Column names
users_columns = ['user_id', 'user_name']

# Create users DataFrame
df_users = spark.createDataFrame(users_data, users_columns)

# Show users table
df_users.show()

+-------+---------+
|user_id|user_name|
+-------+---------+
|    101|     Neel|
|    102|     John|
|    103|    Priya|
+-------+---------+



In [6]:
joined_df = df.join(
    df_users,
    on='user_id',
    how='left'
)

joined_df.show()

+-------+--------+-------+----------+----------------+---------+
|user_id|event_id|country|event_type|session_time_sec|user_name|
+-------+--------+-------+----------+----------------+---------+
|    101|       1|  India|     login|             300|     Neel|
|    102|       2|    USA|  purchase|             900|     John|
|    103|       3|  India|  purchase|            1200|    Priya|
|   NULL|       4| Canada|     login|             200|     NULL|
|    101|       5|  India|    logout|             100|     Neel|
+-------+--------+-------+----------+----------------+---------+



In [7]:
summary_df = df.groupBy(
    'country',
    'event_type'
).agg(
    F.count('event_id').alias('total_events'),
    F.sum('session_time_sec').alias('total_session_time')
)

summary_df.show()

+-------+----------+------------+------------------+
|country|event_type|total_events|total_session_time|
+-------+----------+------------+------------------+
|  India|     login|           1|               300|
|    USA|  purchase|           1|               900|
|  India|  purchase|           1|              1200|
| Canada|     login|           1|               200|
|  India|    logout|           1|               100|
+-------+----------+------------+------------------+



In [8]:
df_with_minutes = df.withColumn(
    'session_minutes',
    F.col('session_time_sec') / 60
)

df_with_minutes.show()

+--------+-------+-------+----------+----------------+------------------+
|event_id|user_id|country|event_type|session_time_sec|   session_minutes|
+--------+-------+-------+----------+----------------+------------------+
|       1|    101|  India|     login|             300|               5.0|
|       2|    102|    USA|  purchase|             900|              15.0|
|       3|    103|  India|  purchase|            1200|              20.0|
|       4|   NULL| Canada|     login|             200|3.3333333333333335|
|       5|    101|  India|    logout|             100|1.6666666666666667|
+--------+-------+-------+----------+----------------+------------------+

